In [1]:
from pathlib import Path
import os
import sys

INPUT_ROOT = Path("/kaggle/input")
COMPETITION_INPUT = Path(
    "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"
)
CODE_INPUT = Path(
    "/kaggle/input/datasets/kagvik35/casmi-vrishin-code-v2"
)
WORKING_ROOT = Path("/kaggle/working")
REPO = WORKING_ROOT / "casmi_repo"

os.environ["PYTHONPATH"] = str(REPO) + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Competition input:", COMPETITION_INPUT)
print("Source input:", CODE_INPUT)
print("Working repository:", REPO)
print("Python:", sys.executable)

Competition input: /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra
Source input: /kaggle/input/datasets/kagvik35/casmi-vrishin-code-v2
Working repository: /kaggle/working/casmi_repo
Python: /usr/bin/python3


In [2]:
from pathlib import Path

competition_root = Path(
    "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"
)
code_root = Path(
    "/kaggle/input/datasets/kagvik35/casmi-vrishin-code-v2"
)

required_competition_files = [
    competition_root / "train.parquet",
    competition_root / "test.parquet",
    competition_root / "sample_submission.csv",
]
required_code_files = [
    code_root / "src/data/build_reference.py",
    code_root / "members/Viki2183/model_analog_coconut.py",
]

for path in required_competition_files + required_code_files:
    print(f"{path}: {path.exists()}")
assert all(path.exists() for path in required_competition_files)
assert all(path.exists() for path in required_code_files)
print("Kaggle inputs verified.")

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet: True
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet: True
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv: True
/kaggle/input/datasets/kagvik35/casmi-vrishin-code-v2/src/data/build_reference.py: True
/kaggle/input/datasets/kagvik35/casmi-vrishin-code-v2/members/Viki2183/model_analog_coconut.py: True
Kaggle inputs verified.


In [3]:
from pathlib import Path
import shutil

source_root = Path(
    "/kaggle/input/datasets/kagvik35/casmi-vrishin-code-v2"
)
repo = Path("/kaggle/working/casmi_repo")

if repo.exists():
    assert (repo / "src/data/build_reference.py").exists(), (
        "Existing working repository is incomplete; refusing to overwrite it."
    )
    assert (repo / "members/Viki2183/model_analog_coconut.py").exists(), (
        "Existing working repository is incomplete; refusing to overwrite it."
    )
    print("Using the already verified working repository:", repo)
else:
    shutil.copytree(source_root, repo)
    print("Copied source code to:", repo)

print("build_reference.py:", (repo / "src/data/build_reference.py").exists())
print("model_analog_coconut.py:", (repo / "members/Viki2183/model_analog_coconut.py").exists())

Copied source code to: /kaggle/working/casmi_repo
build_reference.py: True
model_analog_coconut.py: True


In [4]:
from pathlib import Path
import os

repo = Path("/kaggle/working/casmi_repo")
competition_root = Path(
    "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"
)

train_path = competition_root / "train.parquet"
test_path = competition_root / "test.parquet"
sample_submission_path = competition_root / "sample_submission.csv"

env = os.environ.copy()
env["PYTHONPATH"] = str(repo) + os.pathsep + env.get("PYTHONPATH", "")

assert repo.exists()
assert train_path.exists()
assert test_path.exists()
assert sample_submission_path.exists()

print("repo =", repo)
print("train_path =", train_path)
print("test_path =", test_path)
print("sample_submission_path =", sample_submission_path)
print("PYTHONPATH =", env["PYTHONPATH"])

repo = /kaggle/working/casmi_repo
train_path = /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
test_path = /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet
sample_submission_path = /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
PYTHONPATH = /kaggle/working/casmi_repo:/kaggle/working/casmi_repo:/kaggle/lib/kagglegym:/kaggle/lib:/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra


In [5]:
import importlib.util
import subprocess
import sys

required_modules = ["numpy", "pandas", "scipy", "pyarrow", "sklearn", "rdkit"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]

if "rdkit" in missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "rdkit"],
        check=True,
    )

missing_after = [
    name for name in required_modules if importlib.util.find_spec(name) is None
]
assert not missing_after, f"Missing required modules: {missing_after}"

import numpy
import pandas
import scipy
import pyarrow
import sklearn
import rdkit

print("Verified dependencies:")
for module in [numpy, pandas, scipy, pyarrow, sklearn, rdkit]:
    print(f"  {module.__name__}: {getattr(module, '__version__', 'available')}")

Verified dependencies:
  numpy: 2.1.3
  pandas: 2.3.3
  scipy: 1.16.3
  pyarrow: 23.0.1
  sklearn: 1.6.1
  rdkit: 2026.09.2


In [6]:
from pathlib import Path

repo = Path("/kaggle/working/casmi_repo")
competition_root = Path(
    "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"
)
raw_dir = repo / "data/raw"
raw_dir.mkdir(parents=True, exist_ok=True)

source_files = {
    "train.parquet": competition_root / "train.parquet",
    "test.parquet": competition_root / "test.parquet",
    "sample_submission.csv": competition_root / "sample_submission.csv",
}

for filename, source in source_files.items():
    target = raw_dir / filename
    if target.is_symlink():
        assert target.resolve() == source.resolve(), f"Wrong existing symlink: {target}"
        print("Verified symlink:", target, "->", target.resolve())
    elif target.exists():
        raise FileExistsError(
            f"Refusing to replace existing non-symlink raw file: {target}"
        )
    else:
        target.symlink_to(source)
        print("Created symlink:", target, "->", source)

Created symlink: /kaggle/working/casmi_repo/data/raw/train.parquet -> /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
Created symlink: /kaggle/working/casmi_repo/data/raw/test.parquet -> /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet
Created symlink: /kaggle/working/casmi_repo/data/raw/sample_submission.csv -> /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv


In [7]:
from pathlib import Path
import os
import subprocess
import sys
import pyarrow.parquet as pq

repo = Path("/kaggle/working/casmi_repo")
train_path = Path(
    "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet"
)
processed_dir = repo / "data/processed"
processed_dir.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env["PYTHONPATH"] = str(repo) + os.pathsep + env.get("PYTHONPATH", "")

required = [
    processed_dir / "ref_meta.parquet",
    processed_dir / "structures.parquet",
    processed_dir / "ref_peaks.npz",
]
if not all(path.exists() for path in required):
    command = [
        sys.executable,
        "-m",
        "src.data.build_reference",
        "--train",
        str(train_path),
        "--out",
        str(processed_dir),
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=str(repo), env=env, check=True)
else:
    print("Reference artifacts already exist; reusing them.")

for path in required:
    assert path.exists(), f"Missing processed artifact: {path}"
    print(f"{path.name}: {path.stat().st_size} bytes")

for path in [processed_dir / "ref_meta.parquet", processed_dir / "structures.parquet"]:
    metadata = pq.ParquetFile(path).metadata
    print(f"{path.name}: {metadata.num_rows:,} rows; columns={pq.ParquetFile(path).schema.names}")

Running: /usr/bin/python3 -m src.data.build_reference --train /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet --out /kaggle/working/casmi_repo/data/processed
row group 1/21  47s
row group 2/21  93s
row group 3/21  140s
row group 4/21  185s
row group 5/21  232s
row group 6/21  278s
row group 7/21  324s
row group 8/21  370s
row group 9/21  415s
row group 10/21  459s
row group 11/21  485s
row group 12/21  527s
row group 13/21  539s
row group 14/21  549s
row group 15/21  560s
row group 16/21  570s
row group 17/21  581s
row group 18/21  592s
row group 19/21  602s
row group 20/21  611s
row group 21/21  612s
done: 2,370,323 spectra, 274,506 structures, 48,294,265 peaks, 620s
ref_meta.parquet: 30743686 bytes
structures.parquet: 11479538 bytes
ref_peaks.npz: 386354628 bytes
ref_meta.parquet: 2,370,323 rows; columns=['struct_idx', 'inchikey14', 'ingest_lib', 'ionization_mode', 'adduct', 'precursor_mz', 'peak_start', 'n_peaks']
structures.parquet: 274,506 rows; co

In [8]:
from pathlib import Path
import os
import posixpath
import requests
import subprocess
import sys
import zipfile

repo = Path("/kaggle/working/casmi_repo")
coconut_site_url = "https://coconut.naturalproducts.net"
coconut_download_url = (
    "https://coconut.s3.uni-jena.de/"
    "prod/downloads/2026-11/coconut_csv-11-2026.zip"
)
download_zip = Path("/kaggle/working/coconut_csv-11-2026.zip")
extract_dir = Path("/kaggle/working/coconut_raw")
coconut_parquet = repo / "data/external/coconut/coconut.parquet"
env = os.environ.copy()
env["PYTHONPATH"] = str(repo) + os.pathsep + env.get("PYTHONPATH", "")

if not download_zip.exists():
    print("Downloading from:", coconut_download_url)
    with requests.get(coconut_download_url, stream=True, timeout=300) as response:
        response.raise_for_status()
        with download_zip.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
else:
    print("Reusing downloaded archive:", download_zip)

with zipfile.ZipFile(download_zip) as archive:
    names = [info.filename for info in archive.infolist()]
    unsafe = []
    for name in names:
        normalized = name.replace("\\", "/")
        parts = posixpath.normpath(normalized).split("/")
        if posixpath.isabs(normalized) or ".." in parts:
            unsafe.append(name)
    assert not unsafe, f"Unsafe ZIP entries: {unsafe}"
    if not any(extract_dir.rglob("*.csv")) if extract_dir.exists() else True:
        extract_dir.mkdir(parents=True, exist_ok=True)
        archive.extractall(extract_dir)

csv_files = sorted(extract_dir.rglob("*.csv"))
assert csv_files, f"No COCONUT CSV found; official site: {coconut_site_url}"
coconut_csv = csv_files[0]
print("COCONUT CSV:", coconut_csv)

if not coconut_parquet.exists():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "members.Viki2183.model_analog_coconut",
            "build-coconut",
            "--input",
            str(coconut_csv),
            "--output",
            str(coconut_parquet),
        ],
        cwd=str(repo),
        env=env,
        check=True,
    )
else:
    print("Reusing processed COCONUT table:", coconut_parquet)

assert coconut_parquet.exists()
print("COCONUT parquet:", coconut_parquet, coconut_parquet.stat().st_size, "bytes")

COCONUT CSV: /kaggle/working/coconut_raw/coconut_csv-11-2026.csv


/kaggle/working/casmi_repo/members/Viki2183/model_analog_coconut.py:260: DtypeWarning: Columns (38) have mixed types. Specify dtype option on import or set low_memory=False.
  source_table = pd.read_csv(source)


[coconut] wrote 479694 unique molecules to /kaggle/working/casmi_repo/data/external/coconut/coconut.parquet
COCONUT parquet: /kaggle/working/casmi_repo/data/external/coconut/coconut.parquet 24713992 bytes


In [9]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/kaggle/working/casmi_repo")
env = os.environ.copy()
env["PYTHONPATH"] = str(repo) + os.pathsep + env.get("PYTHONPATH", "")

compile_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(repo / "members/Viki2183/model_analog_coconut.py"),
    ],
    cwd=str(repo),
    env=env,
    text=True,
    capture_output=True,
)
print(compile_result.stdout)
print(compile_result.stderr)
compile_result.check_returncode()

smoke = subprocess.run(
    [
        sys.executable,
        "-m",
        "members.Viki2183.model_analog_coconut",
        "smoke",
    ],
    cwd=str(repo),
    env=env,
    text=True,
    capture_output=True,
)
print(smoke.stdout)
print(smoke.stderr)
smoke.check_returncode()
print("Syntax and smoke tests passed.")



[smoke] helper checks passed


Syntax and smoke tests passed.


In [10]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys

repo = Path("/kaggle/working/casmi_repo")
env = os.environ.copy()
env["PYTHONPATH"] = str(repo) + os.pathsep + env.get("PYTHONPATH", "")
command = [
    sys.executable,
    "-m",
    "members.Viki2183.model_analog_coconut",
    "evaluate",
    "--model",
    "analog",
]
result = subprocess.run(command, cwd=str(repo), env=env, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()

text = result.stdout + "\n" + result.stderr
match = re.search(
    r"\[(?P<model>[^\]]+)\].*?MRR@25 = (?P<mrr>[0-9.eE+-]+)"
    r" \| top-1 (?P<top1>[0-9.eE+-]+)"
    r" \| top-10 (?P<top10>[0-9.eE+-]+)"
    r" \| top-25 (?P<top25>[0-9.eE+-]+)"
    r" \| (?P<molecules>\d+) molecules \| (?P<runtime>\d+)s",
    text,
    flags=re.S,
)
assert match, "Could not parse the actual E601 evaluator output."
record = {
    "experiment": "E601",
    "model": match.group("model"),
    "mrr_at_25": float(match.group("mrr")),
    "top1": float(match.group("top1")),
    "top10": float(match.group("top10")),
    "top25": float(match.group("top25")),
    "molecules": int(match.group("molecules")),
    "runtime_seconds": int(match.group("runtime")),
    "stdout": result.stdout,
}
Path("/kaggle/working/casmi_e601.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))

[analog-search] MRR@25 = 0.9339 | top-1 0.904 | top-10 0.992 | top-25 0.996 | 250 molecules | 406s
{'model': 'analog-search', 'mrr': np.float64(0.9338714285714286), 'top1': np.float64(0.904), 'top10': np.float64(0.992), 'top25': np.float64(0.996)}


{
  "experiment": "E601",
  "model": "analog-search",
  "mrr_at_25": 0.9339,
  "top1": 0.904,
  "top10": 0.992,
  "top25": 0.996,
  "molecules": 250,
  "runtime_seconds": 406,
  "stdout": "[analog-search] MRR@25 = 0.9339 | top-1 0.904 | top-10 0.992 | top-25 0.996 | 250 molecules | 406s\n{'model': 'analog-search', 'mrr': np.float64(0.9338714285714286), 'top1': np.float64(0.904), 'top10': np.float64(0.992), 'top25': np.float64(0.996)}\n"
}


In [11]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys

repo = Path("/kaggle/working/casmi_repo")
env = os.environ.copy()
env["PYTHONPATH"] = str(repo) + os.pathsep + env.get("PYTHONPATH", "")
command = [
    sys.executable,
    "-m",
    "members.Viki2183.model_analog_coconut",
    "evaluate",
    "--model",
    "coconut",
]
result = subprocess.run(command, cwd=str(repo), env=env, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()

text = result.stdout + "\n" + result.stderr
match = re.search(
    r"\[(?P<model>[^\]]+)\].*?MRR@25 = (?P<mrr>[0-9.eE+-]+)"
    r" \| top-1 (?P<top1>[0-9.eE+-]+)"
    r" \| top-10 (?P<top10>[0-9.eE+-]+)"
    r" \| top-25 (?P<top25>[0-9.eE+-]+)"
    r" \| (?P<molecules>\d+) molecules \| (?P<runtime>\d+)s",
    text,
    flags=re.S,
)
assert match, "Could not parse the actual normal E602 evaluator output."
record = {
    "experiment": "E602",
    "mode": "normal",
    "model": match.group("model"),
    "mrr_at_25": float(match.group("mrr")),
    "top1": float(match.group("top1")),
    "top10": float(match.group("top10")),
    "top25": float(match.group("top25")),
    "molecules": int(match.group("molecules")),
    "runtime_seconds": int(match.group("runtime")),
    "stdout": result.stdout,
}
Path("/kaggle/working/casmi_e602_normal.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))

[analog-coconut] MRR@25 = 0.9114 | top-1 0.884 | top-10 0.964 | top-25 0.972 | 250 molecules | 451s
{'model': 'analog-coconut', 'mrr': np.float64(0.9113901587301587), 'top1': np.float64(0.884), 'top10': np.float64(0.964), 'top25': np.float64(0.972)}


{
  "experiment": "E602",
  "mode": "normal",
  "model": "analog-coconut",
  "mrr_at_25": 0.9114,
  "top1": 0.884,
  "top10": 0.964,
  "top25": 0.972,
  "molecules": 250,
  "runtime_seconds": 451,
  "stdout": "[analog-coconut] MRR@25 = 0.9114 | top-1 0.884 | top-10 0.964 | top-25 0.972 | 250 molecules | 451s\n{'model': 'analog-coconut', 'mrr': np.float64(0.9113901587301587), 'top1': np.float64(0.884), 'top10': np.float64(0.964), 'top25': np.float64(0.972)}\n"
}


In [12]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys

repo = Path("/kaggle/working/casmi_repo")
env = os.environ.copy()
env["PYTHONPATH"] = str(repo) + os.pathsep + env.get("PYTHONPATH", "")
command = [
    sys.executable,
    "-m",
    "members.Viki2183.model_analog_coconut",
    "evaluate",
    "--model",
    "coconut",
    "--hide-true",
]
result = subprocess.run(command, cwd=str(repo), env=env, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()

text = result.stdout + "\n" + result.stderr
match = re.search(
    r"\[(?P<model>[^\]]+)\].*?MRR@25 = (?P<mrr>[0-9.eE+-]+)"
    r" \| top-1 (?P<top1>[0-9.eE+-]+)"
    r" \| top-10 (?P<top10>[0-9.eE+-]+)"
    r" \| top-25 (?P<top25>[0-9.eE+-]+)"
    r" \| (?P<molecules>\d+) molecules \| (?P<runtime>\d+)s",
    text,
    flags=re.S,
)
assert match, "Could not parse the actual hide_true E602 evaluator output."
record = {
    "experiment": "E602",
    "mode": "hide_true",
    "model": match.group("model"),
    "mrr_at_25": float(match.group("mrr")),
    "top1": float(match.group("top1")),
    "top10": float(match.group("top10")),
    "top25": float(match.group("top25")),
    "molecules": int(match.group("molecules")),
    "runtime_seconds": int(match.group("runtime")),
    "stdout": result.stdout,
}
Path("/kaggle/working/casmi_e602_hide_true.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))

[analog-coconut (true structure hidden)] MRR@25 = 0.8907 | top-1 0.856 | top-10 0.952 | top-25 0.960 | 250 molecules | 444s
{'model': 'analog-coconut', 'mrr': np.float64(0.8906679365079365), 'top1': np.float64(0.856), 'top10': np.float64(0.952), 'top25': np.float64(0.96)}


{
  "experiment": "E602",
  "mode": "hide_true",
  "model": "analog-coconut (true structure hidden)",
  "mrr_at_25": 0.8907,
  "top1": 0.856,
  "top10": 0.952,
  "top25": 0.96,
  "molecules": 250,
  "runtime_seconds": 444,
  "stdout": "[analog-coconut (true structure hidden)] MRR@25 = 0.8907 | top-1 0.856 | top-10 0.952 | top-25 0.960 | 250 molecules | 444s\n{'model': 'analog-coconut', 'mrr': np.float64(0.8906679365079365), 'top1': np.float64(0.856), 'top10': np.float64(0.952), 'top25': np.float64(0.96)}\n"
}


In [13]:
from pathlib import Path
import json

result_paths = {
    "E601_analog": Path("/kaggle/working/casmi_e601.json"),
    "E602_normal": Path("/kaggle/working/casmi_e602_normal.json"),
    "E602_hide_true": Path("/kaggle/working/casmi_e602_hide_true.json"),
}
records = {}
for name, path in result_paths.items():
    assert path.exists(), f"Missing actual evaluation result: {path}"
    records[name] = json.loads(path.read_text())

summary = {
    "data_paths": {
        "competition": "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra",
        "code": "/kaggle/input/datasets/kagvik35/casmi-vrishin-code-v2",
    },
    "coconut_download_url": (
        "https://coconut.s3.uni-jena.de/"
        "prod/downloads/2026-11/coconut_csv-11-2026.zip"
    ),
    "results": records,
}
json_path = Path("/kaggle/working/casmi_results_summary.json")
markdown_path = Path("/kaggle/working/casmi_results_summary.md")
json_path.write_text(json.dumps(summary, indent=2))

lines = [
    "# CASMI Viki2183 E601/E602 Results",
    "",
    "Generated from the actual evaluator JSON files; no metrics are hard-coded.",
    "",
    "| Experiment | Mode | MRR@25 | Top-1 | Top-10 | Top-25 | Molecules | Runtime (s) |",
    "|---|---|---:|---:|---:|---:|---:|---:|",
]
for name, record in records.items():
    lines.append(
        f"| {record['experiment']} | {record.get('mode', 'analog')} | "
        f"{record['mrr_at_25']:.6f} | {record['top1']:.6f} | "
        f"{record['top10']:.6f} | {record['top25']:.6f} | "
        f"{record['molecules']} | {record['runtime_seconds']} |"
    )
markdown_path.write_text("\n".join(lines) + "\n")
print(json_path.read_text())
print(markdown_path.read_text())

{
  "data_paths": {
    "competition": "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra",
    "code": "/kaggle/input/datasets/kagvik35/casmi-vrishin-code-v2"
  },
  "coconut_download_url": "https://coconut.s3.uni-jena.de/prod/downloads/2026-11/coconut_csv-11-2026.zip",
  "results": {
    "E601_analog": {
      "experiment": "E601",
      "model": "analog-search",
      "mrr_at_25": 0.9339,
      "top1": 0.904,
      "top10": 0.992,
      "top25": 0.996,
      "molecules": 250,
      "runtime_seconds": 406,
      "stdout": "[analog-search] MRR@25 = 0.9339 | top-1 0.904 | top-10 0.992 | top-25 0.996 | 250 molecules | 406s\n{'model': 'analog-search', 'mrr': np.float64(0.9338714285714286), 'top1': np.float64(0.904), 'top10': np.float64(0.992), 'top25': np.float64(0.996)}\n"
    },
    "E602_normal": {
      "experiment": "E602",
      "mode": "normal",
      "model": "analog-coconut",
      "mrr_at_25": 0.9114,
      "top1": 0.884,
      "top10": 0.964,
      "top25": 0.